### Campaign Performance Analysis - Nova Mart

Following up on our recent discussion, our client, Nova Mart, a major retail chain in southern India, has provided us with data from their Diwali 2023 and Sankranti 2024 promotional campaigns. They ran massive promotions across all 50 stores on their Nova-branded products. Your task is to analyze this data using the Pandas library to provide insights into campaign performance.

Specifically, the Sales Director wants answers to key business questions regarding the promotions' effectiveness and their impact on sales across various stores and product categories.

Click on the "Download Files" button above to download all the files. Your specific task is described in client_requests.pdf. Be sure to check meta_data.md to understand the dataset thoroughly.

If you have any follow-up questions on the task, you can ask your seniors here: [discord link]

But they are busy, try to solve it on your own as much as possible.

Good luck with your task!

1. The operations team wants to ensure the integrity of the events data by removing
duplicates. Check for and remove duplicate rows in the events dataframe based
on store_id, campaign_id, and product_code. How many duplicate rows were
removed?

In [1]:
import pandas as pd

# 1. Baca dataset fact_events
fact_events = pd.read_csv('resources_1/fact_events.csv')

# 2. Hitung jumlah baris awal sebelum pembersihan
initial_rows = len(fact_events)

# 3. Hitung jumlah baris duplikat berdasarkan kombinasi kolom store_id, campaign_id, dan product_code
duplicate_count = fact_events.duplicated(
    subset=['store_id', 'campaign_id', 'product_code']
).sum()

# 4. Hapus baris duplikat dan simpan ke dataframe baru (gunakan .copy() agar aman untuk proses selanjutnya)
fact_events_cleaned = fact_events.drop_duplicates(
    subset=['store_id', 'campaign_id', 'product_code']
).copy()

# 5. Tampilkan hasil verifikasi pembersihan duplikat
print(f"Jumlah baris awal: {initial_rows}")
print(f"Jumlah baris setelah dibersihkan: {len(fact_events_cleaned)}")
print(f"Jumlah baris duplikat yang dihapus: {duplicate_count}")

Jumlah baris awal: 1510
Jumlah baris setelah dibersihkan: 1500
Jumlah baris duplikat yang dihapus: 10


2. How many cities have more than 5 stores?

In [2]:
# 1. Baca dataset dim_stores untuk melihat informasi toko dan lokasinya
dim_stores = pd.read_csv('resources_1/dim_stores.csv')

# 2. Hitung jumlah toko di setiap kota menggunakan value_counts()
store_counts_per_city = dim_stores['city'].value_counts()

# 3. Filter kota-kota yang memiliki lebih dari 5 toko (jumlah toko > 5)
cities_more_than_5 = store_counts_per_city[store_counts_per_city > 5]

# 4. Hitung berapa banyak kota yang memenuhi kriteria tersebut
total_cities_more_than_5 = len(cities_more_than_5)

# 5. Tampilkan rincian kota beserta jumlah tokonya dan total banyaknya kota
print("Daftar kota yang memiliki lebih dari 5 toko:")
print(cities_more_than_5)
print(f"\nJumlah kota yang memiliki lebih dari 5 toko: {total_cities_more_than_5}")

Daftar kota yang memiliki lebih dari 5 toko:
city
Bengaluru    10
Chennai       8
Hyderabad     7
Name: count, dtype: int64

Jumlah kota yang memiliki lebih dari 5 toko: 3


3. The sales team has identified missing values in the quantity_sold(before_promo)
data. Estimate these values using the median quantity sold before the promotion.
How many missing values were filled, and what is the median used for
imputation?

In [3]:
# 1. Cek jumlah missing values (nilai NaN/kosong) pada kolom 'quantity_sold(before_promo)'
missing_count = fact_events_cleaned['quantity_sold(before_promo)'].isna().sum()

# 2. Hitung nilai median dari kolom 'quantity_sold(before_promo)' (mengabaikan nilai NaN secara default)
median_quantity_sold = fact_events_cleaned['quantity_sold(before_promo)'].median()

# 3. Lakukan imputasi (mengisi nilai NaN) dengan nilai median menggunakan fungsi .fillna()
fact_events_cleaned['quantity_sold(before_promo)'] = fact_events_cleaned['quantity_sold(before_promo)'].fillna(median_quantity_sold)

# 4. Verifikasi kembali untuk memastikan tidak ada lagi missing values
remaining_missing = fact_events_cleaned['quantity_sold(before_promo)'].isna().sum()

# 5. Tampilkan ringkasan hasil imputasi
print(f"Jumlah missing values yang ditemukan dan diisi: {missing_count}")
print(f"Nilai median yang digunakan untuk imputasi: {median_quantity_sold}")
print(f"Jumlah missing values setelah imputasi: {remaining_missing}")

Jumlah missing values yang ditemukan dan diisi: 20
Nilai median yang digunakan untuk imputasi: 78.0
Jumlah missing values setelah imputasi: 0


4. Identify the product category with the lowest base price before the promotion.

In [4]:
# 1. Baca dataset dim_products yang memuat kategori produk
dim_products = pd.read_csv('resources_1/dim_products.csv')

# 2. Gabungkan (merge) dataframe fact_events_cleaned dengan dim_products berdasarkan 'product_code'
events_with_products = fact_events_cleaned.merge(dim_products, on='product_code', how='left')

# 3. Cari harga dasar terendah (minimum base_price) untuk setiap kategori produk sebelum promo
lowest_price_by_category = events_with_products.groupby('category')['base_price(before_promo)'].min().sort_values()

# 4. Ambil kategori dengan harga terendah dan nilainya
lowest_category = lowest_price_by_category.index[0]
lowest_price = lowest_price_by_category.iloc[0]

# 5. Tampilkan ringkasan harga minimum per kategori dan kategori dengan harga terendah
print("Harga dasar terendah (minimum base price) per kategori sebelum promosi:")
print(lowest_price_by_category)
print(f"\nKategori produk dengan base price terendah sebelum promosi adalah: {lowest_category} (Harga: {lowest_price})")

Harga dasar terendah (minimum base price) per kategori sebelum promosi:
category
Personal Care          50
Home Care              55
Grocery & Staples     156
Home Appliances       350
Combo1               3000
Name: base_price(before_promo), dtype: int64

Kategori produk dengan base price terendah sebelum promosi adalah: Personal Care (Harga: 50)


5. What is the total quantity sold after the promotion for the BOGOF promo type
during the Diwali campaign?

In [5]:
# 1. Baca dataset dim_campaigns untuk melihat detail nama kampanye
dim_campaigns = pd.read_csv('resources_1/dim_campaigns.csv')

# 2. Gabungkan dataframe fact_events_cleaned dengan dim_campaigns berdasarkan 'campaign_id'
events_with_campaigns = fact_events_cleaned.merge(dim_campaigns, on='campaign_id', how='left')

# 3. Filter data berdasarkan kriteria: kampanye 'Diwali' dan promo_type 'BOGOF'
diwali_bogof_events = events_with_campaigns[
    (events_with_campaigns['campaign_name'] == 'Diwali') & 
    (events_with_campaigns['promo_type'] == 'BOGOF')
]

# 4. Hitung total kuantitas yang terjual setelah promosi (quantity_sold(after_promo))
total_quantity_sold_after = diwali_bogof_events['quantity_sold(after_promo)'].sum()

# 5. Tampilkan hasil perhitungan
print(f"Total kuantitas yang terjual (quantity sold) setelah promosi untuk tipe BOGOF pada kampanye Diwali: {total_quantity_sold_after:,} unit")

Total kuantitas yang terjual (quantity sold) setelah promosi untuk tipe BOGOF pada kampanye Diwali: 34,461 unit


6. Which store recorded the highest quantity sold after the promotion during the
Diwali campaign?

In [6]:
# 1. Pastikan dataset dim_stores sudah terbaca untuk mendapatkan informasi kota jika diperlukan
dim_stores = pd.read_csv('resources_1/dim_stores.csv')

# 2. Gabungkan data events_with_campaigns dengan dim_stores berdasarkan 'store_id'
events_full = events_with_campaigns.merge(dim_stores, on='store_id', how='left')

# 3. Filter transaksi khusus untuk kampanye 'Diwali'
diwali_events = events_full[events_full['campaign_name'] == 'Diwali']

# 4. Kelompokkan berdasarkan 'store_id' (dan 'city') lalu hitung total 'quantity_sold(after_promo)'
store_sales_diwali = diwali_events.groupby(['store_id', 'city'])['quantity_sold(after_promo)'].sum().reset_index()

# 5. Urutkan toko berdasarkan penjualan tertinggi ke terendah
store_sales_diwali_sorted = store_sales_diwali.sort_values(by='quantity_sold(after_promo)', ascending=False)

# 6. Ambil data toko peringkat pertama (tertinggi)
top_store = store_sales_diwali_sorted.iloc[0]

# 7. Tampilkan hasil 5 toko teratas dan toko tertinggi
print("5 Toko dengan penjualan tertinggi setelah promo selama kampanye Diwali:")
print(store_sales_diwali_sorted.head(5).to_string(index=False))
print(f"\nToko dengan kuantitas penjualan tertinggi: {top_store['store_id']} di {top_store['city']} dengan {top_store['quantity_sold(after_promo)']:,} unit")

5 Toko dengan penjualan tertinggi setelah promo selama kampanye Diwali:
store_id      city  quantity_sold(after_promo)
 STCHE-4   Chennai                        5013
 STBLR-7 Bengaluru                        4893
 STBLR-6 Bengaluru                        4857
 STCHE-7   Chennai                        4779
 STMYS-1    Mysuru                        4779

Toko dengan kuantitas penjualan tertinggi: STCHE-4 di Chennai dengan 5,013 unit


7. Understand which campaigns had the most successful outcomes. Compare the
total quantities sold before and after the promotions for the Sankranti and Diwali
campaigns. Which campaign saw a greater increase in sales?

In [7]:
# 1. Kelompokkan data per kampanye dan hitung total kuantitas sebelum dan sesudah promosi
campaign_comparison = events_full.groupby('campaign_name').agg(
    total_qty_before=('quantity_sold(before_promo)', 'sum'),
    total_qty_after=('quantity_sold(after_promo)', 'sum')
).reset_index()

# 2. Hitung kenaikan absolut penjualan (selisih unit terjual setelah vs sebelum promo)
campaign_comparison['sales_increase_units'] = (
    campaign_comparison['total_qty_after'] - campaign_comparison['total_qty_before']
)

# 3. Hitung persentase kenaikan penjualan (% Increase)
campaign_comparison['pct_increase'] = (
    (campaign_comparison['sales_increase_units'] / campaign_comparison['total_qty_before']) * 100
)

# 4. Tentukan kampanye mana yang mengalami peningkatan penjualan lebih besar
best_campaign = campaign_comparison.loc[campaign_comparison['sales_increase_units'].idxmax()]

# 5. Tampilkan tabel perbandingan dan kesimpulan
print("Perbandingan performa penjualan antara kampanye Sankranti dan Diwali:")
print(campaign_comparison.to_string(index=False))
print(f"\nKampanye dengan peningkatan penjualan lebih besar: {best_campaign['campaign_name']} (Kenaikan: {best_campaign['sales_increase_units']:,.0f} unit atau {best_campaign['pct_increase']:.2f}%)")

Perbandingan performa penjualan antara kampanye Sankranti dan Diwali:
campaign_name  total_qty_before  total_qty_after  sales_increase_units  pct_increase
       Diwali          109756.0           183404               73648.0     67.101571
    Sankranti           97894.0           252069              154175.0    157.491777

Kampanye dengan peningkatan penjualan lebih besar: Sankranti (Kenaikan: 154,175 unit atau 157.49%)


8. Which product recorded the highest Incremental Revenue Percentage (IR%)
during the Sankranti campaign? What is the IR% for this product?

In [8]:
# 1. Filter data khusus untuk kampanye 'Sankranti'
sankranti_events = events_full[events_full['campaign_name'] == 'Sankranti'].copy()

# 2. Hitung total pendapatan (Revenue) sebelum dan sesudah promosi untuk setiap baris transaksi
# Formula: Revenue = base_price * quantity_sold
sankranti_events['revenue_before'] = (
    sankranti_events['base_price(before_promo)'] * sankranti_events['quantity_sold(before_promo)']
)
sankranti_events['revenue_after'] = (
    sankranti_events['base_price(after_promo)'] * sankranti_events['quantity_sold(after_promo)']
)

# 3. Gabungkan dengan dim_products untuk mendapatkan 'product_name'
sankranti_with_products = sankranti_events.merge(dim_products, on='product_code', how='left')

# 4. Agregasi total revenue sebelum dan sesudah promo untuk setiap produk
product_revenue = sankranti_with_products.groupby(['product_code', 'product_name']).agg(
    total_revenue_before=('revenue_before', 'sum'),
    total_revenue_after=('revenue_after', 'sum')
).reset_index()

# 5. Hitung Incremental Revenue Percentage (IR%)
# Formula: IR% = ((Revenue_after - Revenue_before) / Revenue_before) * 100
product_revenue['IR%'] = (
    ((product_revenue['total_revenue_after'] - product_revenue['total_revenue_before']) / 
     product_revenue['total_revenue_before']) * 100
)

# 6. Urutkan berdasarkan IR% tertinggi
product_revenue_sorted = product_revenue.sort_values(by='IR%', ascending=False)
highest_ir_product = product_revenue_sorted.iloc[0]

# 7. Tampilkan tabel 5 produk dengan IR% tertinggi dan produk nomor 1
print("5 Produk dengan IR% tertinggi selama kampanye Sankranti:")
print(product_revenue_sorted.head(5).to_string(index=False))
print(f"\nProduk dengan IR% tertinggi: {highest_ir_product['product_name']} ({highest_ir_product['product_code']}) dengan IR% sebesar {highest_ir_product['IR%']:.2f}%")

5 Produk dengan IR% tertinggi selama kampanye Sankranti:
product_code                         product_name  total_revenue_before  total_revenue_after       IR%
         P03              Atliq_Suflower_Oil (1L)             3189600.0              6118500 91.826561
         P15 Atliq_Home_Essential_8_Product_Combo            16185000.0             31027500 91.705283
         P13          Atliq_High_Glo_15W_LED_Bulb             1740550.0              3303125 89.774784
         P14       Atliq_waterproof_Immersion_Rod             4542060.0              8534850 87.907029
         P04         Atliq_Farm_Chakki_Atta (1KG)             6813550.0             12779800 87.564485

Produk dengan IR% tertinggi: Atliq_Suflower_Oil (1L) (P03) dengan IR% sebesar 91.83%


9. Which store in Visakhapatnam recorded the lowest Incremental Sold Units
Percentage (ISU%) during the Diwali campaign? What is the ISU% for that
store?

In [9]:
# 1. Filter data khusus untuk kota 'Visakhapatnam' dan kampanye 'Diwali'
visakh_diwali_events = events_full[
    (events_full['city'] == 'Visakhapatnam') & 
    (events_full['campaign_name'] == 'Diwali')
]

# 2. Agregasikan kuantitas terjual sebelum dan sesudah promosi untuk setiap toko di Visakhapatnam
store_isu = visakh_diwali_events.groupby('store_id').agg(
    total_qty_before=('quantity_sold(before_promo)', 'sum'),
    total_qty_after=('quantity_sold(after_promo)', 'sum')
).reset_index()

# 3. Hitung Incremental Sold Units Percentage (ISU%)
# Formula: ISU% = ((Qty_after - Qty_before) / Qty_before) * 100
store_isu['ISU%'] = (
    ((store_isu['total_qty_after'] - store_isu['total_qty_before']) / 
     store_isu['total_qty_before']) * 100
)

# 4. Urutkan dari ISU% terendah (ascending)
store_isu_sorted = store_isu.sort_values(by='ISU%', ascending=True)
lowest_isu_store = store_isu_sorted.iloc[0]

# 5. Tampilkan tabel performa toko di Visakhapatnam dan toko dengan ISU% terendah
print("Performa ISU% toko di Visakhapatnam selama kampanye Diwali:")
print(store_isu_sorted.to_string(index=False))
print(f"\nToko dengan ISU% terendah di Visakhapatnam: {lowest_isu_store['store_id']} dengan ISU% sebesar {lowest_isu_store['ISU%']:.2f}%")

Performa ISU% toko di Visakhapatnam selama kampanye Diwali:
store_id  total_qty_before  total_qty_after      ISU%
 STVSK-3            1780.0             2656 49.213483
 STVSK-4            1926.0             2908 50.986501
 STVSK-1            1903.0             3078 61.744614
 STVSK-2            1701.0             2860 68.136390
 STVSK-0            1768.0             3005 69.966063

Toko dengan ISU% terendah di Visakhapatnam: STVSK-3 dengan ISU% sebesar 49.21%


10. Which promo type had both a negative Incremental Revenue Percentage (IR%)
and Incremental Sold Units Percentage (ISU%) during the Sankranti campaign?

In [10]:
# 1. Gunakan data kampanye Sankranti yang sudah memiliki perhitungan 'revenue_before' dan 'revenue_after'
# 2. Kelompokkan berdasarkan 'promo_type' untuk menghitung total unit dan total revenue
promo_performance = sankranti_events.groupby('promo_type').agg(
    total_qty_before=('quantity_sold(before_promo)', 'sum'),
    total_qty_after=('quantity_sold(after_promo)', 'sum'),
    total_rev_before=('revenue_before', 'sum'),
    total_rev_after=('revenue_after', 'sum')
).reset_index()

# 3. Hitung ISU% (Incremental Sold Units %)
promo_performance['ISU%'] = (
    ((promo_performance['total_qty_after'] - promo_performance['total_qty_before']) / 
     promo_performance['total_qty_before']) * 100
)

# 4. Hitung IR% (Incremental Revenue %)
promo_performance['IR%'] = (
    ((promo_performance['total_rev_after'] - promo_performance['total_rev_before']) / 
     promo_performance['total_rev_before']) * 100
)

# 5. Filter tipe promo yang memiliki ISU% < 0 DAN IR% < 0 (keduanya bernilai negatif)
negative_promo = promo_performance[
    (promo_performance['ISU%'] < 0) & 
    (promo_performance['IR%'] < 0)
]

# 6. Tampilkan ringkasan semua promo_type dan promo yang keduanya negatif
print("Performa semua promo_type selama kampanye Sankranti:")
print(promo_performance[['promo_type', 'ISU%', 'IR%']].to_string(index=False))
print(f"\nTipe promo yang memiliki IR% DAN ISU% bernilai negatif: {negative_promo['promo_type'].iloc[0]}")

Performa semua promo_type selama kampanye Sankranti:
  promo_type       ISU%        IR%
     25% OFF -19.603090 -39.329552
     33% OFF  41.146205  -5.903300
     50% OFF  37.047854 -31.075563
500 Cashback 130.046339  91.705283
       BOGOF 278.044037  88.327227

Tipe promo yang memiliki IR% DAN ISU% bernilai negatif: 25% OFF


Key Metrics:
</br>
● IR% (Incremental Revenue): IR% measures the percentage change in revenue
after a promotion compared to the revenue before the promotion. It helps assess
how effective a promotion was in driving revenue growth.
</br>
● ISU% (Incremental Sold Units): ISU% calculates the percentage change in the
number of units sold after a promotion compared to the units sold before the
promotion. It indicates the effectiveness of a promotion in boosting sales volume